In [2]:
import sqlite3
import numpy
import csv
import os

In [3]:
ja2label = {'ㄱ':0, 'ㄲ':1, 'ㄴ':2, 'ㄷ':3, 'ㄸ':4, 'ㄹ':5, 'ㅁ':6, 'ㅂ':7, 'ㅃ':8,
'ㅅ':9, 'ㅆ':10, 'ㅇ':11,  'ㅈ':12, 'ㅉ':13, 'ㅊ':14, 'ㅋ':15, 'ㅌ':16,  'ㅍ':17, 'ㅎ':18}

mo2label = {'ㅏ':0, 'ㅐ':1, 'ㅑ':2, 'ㅒ':3, 'ㅓ':4, 'ㅔ':5, 'ㅕ':6, 'ㅖ':7, 'ㅗ':8, 'ㅘ':9, 
'ㅙ':10, 'ㅚ':11, 'ㅛ':12, 'ㅜ':13, 'ㅝ':14, 'ㅞ':15, 'ㅟ':16, 'ㅠ':17, 'ㅡ':18, 'ㅢ':19, 'ㅣ':20}

ba2label = {None:0, 'ㄱ':1, 'ㄲ':2, 'ㄳ':3, 'ㄴ':4, 'ㄵ':5, 'ㄶ':6, 'ㄷ':7, 'ㄹ':8, 'ㄺ':9,
'ㄻ':10, 'ㄼ':11, 'ㄽ':12, 'ㄾ':13, 'ㄿ':14, 'ㅀ':15, 'ㅁ':16, 'ㅂ':17, 'ㅄ':18, 'ㅅ':19,
'ㅆ':20, 'ㅇ':21, 'ㅈ':22, 'ㅊ':23, 'ㅋ':24, 'ㅌ':25, 'ㅍ':26, 'ㅎ':27}

label2ja = {0: 'ㄱ', 1: 'ㄲ', 2: 'ㄴ', 3: 'ㄷ', 4: 'ㄸ', 5: 'ㄹ',
            6: 'ㅁ', 7: 'ㅂ', 8: 'ㅃ', 9: 'ㅅ', 10: 'ㅆ', 11: 'ㅇ',
            12: 'ㅈ', 13: 'ㅉ', 14: 'ㅊ', 15: 'ㅋ', 16: 'ㅌ', 17: 'ㅍ', 18: 'ㅎ'}

label2mo = {0: 'ㅏ', 1: 'ㅐ', 2: 'ㅑ', 3: 'ㅒ', 4: 'ㅓ', 5: 'ㅔ',
            6: 'ㅕ', 7: 'ㅖ', 8: 'ㅗ', 9: 'ㅘ', 10: 'ㅙ', 11: 'ㅚ',
            12: 'ㅛ', 13: 'ㅜ', 14: 'ㅝ', 15: 'ㅞ', 16: 'ㅟ', 17: 'ㅠ',
            18: 'ㅡ', 19: 'ㅢ', 20: 'ㅣ'}

label2ba = {0: None, 1: 'ㄱ', 2: 'ㄲ', 3: 'ㄳ', 4: 'ㄴ', 5: 'ㄵ',
            6: 'ㄶ', 7: 'ㄷ', 8: 'ㄹ', 9: 'ㄺ', 10: 'ㄻ', 11: 'ㄼ',
            12: 'ㄽ', 13: 'ㄾ', 14: 'ㄿ', 15: 'ㅀ', 16: 'ㅁ', 17: 'ㅂ',
            18: 'ㅄ', 19: 'ㅅ', 20: 'ㅆ', 21: 'ㅇ', 22: 'ㅈ', 23: 'ㅊ',
            24: 'ㅋ', 25: 'ㅌ', 26: 'ㅍ', 27: 'ㅎ'}

ASC2label = {'0' : 0, '1': 1, '2' : 2, '3' : 3, '4' : 4, '5' : 5, '6' : 6, '7' : 7, '8' : 8, '9' : 9,
             'A' : 10, 'B': 11, 'C' : 12, 'D' : 13, 'E' : 14, 'F' : 15, 'G' : 16, 'H' : 17, 'I' : 18, 'J' : 19, 'K' : 20, 'L' : 21, 'M' : 22,
             'N' : 23, 'O': 24, 'P' : 25, 'Q' : 26, 'R' : 27, 'S' : 28, 'T' : 29, 'U' : 30, 'V' : 31, 'W' : 32, 'X' : 33, 'Y' : 34, 'Z' : 35,
             'a' : 36, 'b': 37, 'c' : 38, 'd' : 39, 'e' : 40, 'f' : 41, 'g' : 42, 'h' : 43, 'i' : 44, 'j' : 45, 'k' : 46, 'l' : 47, 'm' : 48,
             'n' : 49, 'o': 50, 'p' : 51, 'q' : 52, 'r' : 53, 's' : 54, 't' : 55, 'u' : 56, 'v' : 57, 'w' : 58, 'x' : 59, 'y' : 60, 'z' : 61,
             ':' : 62, '#': 63, '@' : 64, '(' : 65, ')' : 66, '-' : 67}

label2ASC = {0 : '0', 1 : '1', 2 : '2', 3 : '3', 4 : '4', 5 : '5', 6 : '6', 7 : '7', 8 : '8', 9 : '9',
             10 : 'A', 11 : 'B', 12 : 'C', 13 : 'D', 14 : 'E', 15 : 'F', 16 : 'G', 17 : 'H', 18 : 'I', 19 : 'J', 20 : 'K', 21: 'L', 22 :'M',
             23 : 'N', 24 : 'O', 25 : 'P', 26 : 'Q', 27 : 'R', 28 : 'S', 29 : 'T', 30 :'U', 31: 'V', 32 : 'W', 33 : 'X', 34 : 'Y', 35 : 'Z',
             36 : 'a', 37 : 'b', 38 : 'c', 39 : 'd', 40 : 'e', 41 : 'f', 42 : 'g', 43 : 'h', 44 : 'i', 45 : 'j', 46 : 'k', 47 : 'l', 48 : 'm',
             49 : 'n', 50 : 'o', 51 : 'p', 52 : 'q', 53 : 'r', 54 : 's', 55 : 't', 56 : 'u', 57 : 'v', 58 : 'w', 59 : 'x', 60 : 'y', 61 : 'z',
             62 : ':', 63 : '#', 64 : '@', 65 : '(', 66 : ')', 67 : '-'
             }

In [17]:
def MergeDataset(baseDir, outPath = ""):
    cmdString = "cat"
    commonFileName = 'flt.csv'
    for file in baseDir:
        cmdString = cmdString + " " + file + "/" + commonFileName
    cmdString = cmdString + " > " + outPath
    
    print(cmdString)
    #os.system("cat text1.txt text2.text > merge.txt")
    os.system(cmdString)

In [28]:
def make_han_db(csv_dataset_path, db_path = './DB/HAN_DB.db'):
    han_conn = sqlite3.connect(db_path)
    han_cur = han_conn.cursor()
    han_csvFile = open(csv_dataset_path, 'r', encoding='utf-8')
    han_reader = csv.reader(han_csvFile)
    
    data = list(tuple())
    for han_line in han_reader:
        data.append((han_line[0], int(han_line[1]), int(han_line[2]), int(han_line[3])))
                
    han_cur.execute("""CREATE TABLE IF NOT EXISTS han_training_db(
    image_path TEXT,
    cho INTEGER,
    jung INTEGER,
    jong INTEGER
    )""")

    han_cur.executemany('INSERT INTO han_training_db (image_path, cho, jung, jong) VALUES (?, ?, ?, ?)', data)
    
    han_conn.commit()
    han_conn.close()

def make_asc_db(csv_dataset_path, db_path = './DB/ASCII_DB.db'):
    asc_conn = sqlite3.connect(db_path)
    asc_cur = asc_conn.cursor()
    asc_csvFile = open(csv_dataset_path, 'r', encoding='utf-8')
    asc_reader = csv.reader(asc_csvFile)
    
    data = list(tuple())
    for asc_line in asc_reader:
        data.append((asc_line[0], int(asc_line[1])))
        
    asc_cur.execute("""CREATE TABLE IF NOT EXISTS asc_training_db(
    image_path TEXT,
    char_val INTEGER
    )""")

    asc_cur.executemany('INSERT INTO asc_training_db (image_path, char_val) VALUES (?, ?)', data)
    
    asc_conn.commit()
    asc_conn.close()
    
asc_train_datasetList = ['/mnt/d/linData/synth_asc1/train',
               '/mnt/d/linData/synth_asc3/train',
               '/mnt/d/linData/synth_asc4/train',
               '/mnt/d/linData/synth_asc5/train',
               '/mnt/d/linData/synth_asc6/train',
               '/mnt/d/linData/synth_asc7/train',
               '/mnt/d/linData/synth_asc8/train'
               ]

asc_csv_dataset_path = "/root/Data/hangul/dataset/ASC_train.csv"
MergeDataset(asc_train_datasetList, asc_csv_dataset_path)
make_asc_db(asc_csv_dataset_path)


make_han_db('/root/Data/hangul/dataset/reduce_train_shuffle.csv')
    

cat /mnt/d/linData/synth_asc1/train/flt.csv /mnt/d/linData/synth_asc3/train/flt.csv /mnt/d/linData/synth_asc4/train/flt.csv /mnt/d/linData/synth_asc5/train/flt.csv /mnt/d/linData/synth_asc6/train/flt.csv /mnt/d/linData/synth_asc7/train/flt.csv /mnt/d/linData/synth_asc8/train/flt.csv > /root/Data/hangul/dataset/ASC_train.csv


In [ ]:
def getSpecificExtensionFiles(path, extension):
    out = []
    for (path, dir, files) in os.walk(path):
        for filename in files:
            ext = os.path.splitext(filename)[-1]
            if ext == extension:
                #print("%s/%s" % (path, filename))
                out.append(path + "/" + filename)
    return out

han_train_conn = sqlite3.connect('./DB/HAN_DB.db')
han_train_cur = han_train_conn.cursor()

def get_dataset_fromCsv():
    cnt = 0
    
    fontFiles = getSpecificExtensionFiles("/root/Data/font/clova-all", ".ttf")
    han_train_cur.execute("""SELECT * FROM asc_training_db ORDER BY RANDOM() LIMIT 8000
            """)
    limit_data = han_train_cur.fetchall()
    
    for path, v_cho, v_jung, v_jong in limit_data:
        if not os.path.exists(path):
            #print("File doesnt exist, File : ", imgFile)
            continue
        
        img = Image.open(path)
        img = img.resize((64,64))
        
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = np.array(img)
        img = np.expand_dims(img, axis=0)
        
        label1 = np.expand_dims(np.array(int(v_cho)), axis=0)
        label2 = np.expand_dims(np.array(int(v_jung)), axis=0)
        label3 = np.expand_dims(np.array(int(v_jong)), axis=0)
        
        yield img, (label1, label2, label3)
        
        fontidx = random.randrange(0, fontNum)
        yield getFontImage(fontFiles[fontidx], 10)

        if (cnt > DATA_SIZE): break
        else                : cnt += 1

In [ ]:
def get_dataset_fromCSV_m():
    os.system("shuf /root/Data/hangul/dataset/deDup_reduce.csv > /root/Data/hangul/dataset/reduce_train_shuffle.csv")
    han_csvFile = open("/root/Data/hangul/dataset/reduce_train_shuffle.csv", 'r', encoding='utf-8')
    han_reader = csv.reader(han_csvFile)
    
    os.system("shuf /root/Data/hangul/dataset/ASC_train.csv > /root/Data/hangul/dataset/ASC_train_shuffle.csv")
    asc_csvFile = open("/root/Data/hangul/dataset/ASC_train_shuffle.csv", 'r', encoding='utf-8')
    asc_reader = csv.reader(asc_csvFile)
    
    han_label_idx = len(ASC2label)
    cnt = 0
    
    for han_line, asc_line in zip(han_reader, asc_reader):
        asc_imgFile = asc_line[0]
        
        if not os.path.exists(asc_imgFile):
            continue

        asc_img = Image.open(asc_imgFile)
        asc_img = asc_img.resize((64,64))
        
        asc_img = tf.image.convert_image_dtype(asc_img, tf.float32)
        asc_img = np.array(asc_img)
        asc_img = np.expand_dims(asc_img, axis=0)
        
        asc_label = np.expand_dims(np.array(int(asc_line[1])), axis=0)
        # asc_label = dup_label[int(asc_line[1])]
        # print("new : ", asc_label)
        # print("origin : ", np.expand_dims(np.array(int(asc_line[1])), axis=0))
        
        cnt += 1
        yield asc_img, (asc_label)
        
        if (cnt % 20 != 0):
            continue
        
        han_imgFile = han_line[0]
        
        if not os.path.exists(han_imgFile):
            continue
        
        han_img = Image.open(han_imgFile)
        han_img = han_img.resize((64,64))
        
        han_img = tf.image.convert_image_dtype(han_img, tf.float32)
        han_img = np.array(han_img)
        han_img = np.expand_dims(han_img, axis = 0)
        
        han_label = np.expand_dims(np.array(int(han_label_idx)), axis = 0)
        #han_label = dup_label[int(han_line[1])]
        
        cnt += 1
        yield han_img, (han_label)
        
              
        if (cnt > DATA_SIZE): break